# 01 · Data and book reconstruction

LOBSTER sample day 2012-06-21 for AMZN, AAPL, GOOG, INTC, MSFT. This notebook
loads the Parquet files, replays a day through the C++ book, checks the
reconstruction against LOBSTER's own orderbook file, and looks at the L1 series.

Key idea from `docs/LOBSTER_FORMAT.md`: a *level-10* message file omits every event
outside the top 10 levels, so a pure message replay drifts once a level leaves and
re-enters the window. Three modes are compared below.

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "python"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import lob
from lob import data as D
from lob.reconstruct import replay_day
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
print("lob", lob.__version__, "| tickers:", [d.ticker for d in D.find_days()])

In [ ]:
msgs = D.load_messages("AMZN")
df = msgs.to_frame()
print(len(df), "messages;", df.ts_ns.min()/1e9, "->", df.ts_ns.max()/1e9, "s after midnight")
df.type.value_counts().sort_index().rename({1:"1 new",2:"2 partial cancel",3:"3 delete",4:"4 exec visible",5:"5 exec hidden",6:"6 cross",7:"7 halt"})

In [ ]:
from lob.validate import validate
reports = {t: validate(t) for t in ["AMZN","AAPL","GOOG","INTC","MSFT"]}
rows = [{"ticker": t, "mode": m, "top10_exact_%": r.exact_pct, "L1_exact_%": r.l1_pct, "first_mismatch": r.first_mismatch}
        for t, rep in reports.items() for m, r in rep.modes.items()]
pd.DataFrame(rows).pivot(index="ticker", columns="mode", values=["L1_exact_%","top10_exact_%"]).round(2)

`snapshot_resync` is exact by construction; its counters say how much the
10-level truncation hides (levels entering/leaving the window, quantities changed off-file).

In [ ]:
pd.DataFrame({t: {k: v for k, v in rep.modes["snapshot_resync"].stats.items() if k.startswith("resync") or k in ("unknown_id","priority_purges")}
              for t, rep in reports.items()}).T

In [ ]:
feat, raw = replay_day("AMZN", depth_k=10)
t = pd.to_datetime(feat.ts_ns, unit="ns")
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ax[0].plot(t, feat.best_bid/1e4, lw=.5, label="bid"); ax[0].plot(t, feat.best_ask/1e4, lw=.5, label="ask"); ax[0].legend(); ax[0].set_title("AMZN L1")
ax[1].plot(t, feat.spread_ticks, lw=.4); ax[1].set_ylabel("spread (ticks)")
ax[2].plot(t, feat.bid_qty, lw=.4, label="bid qty"); ax[2].plot(t, feat.ask_qty, lw=.4, label="ask qty"); ax[2].legend(); ax[2].set_ylabel("shares")
fig.tight_layout(); fig.savefig(RESULTS/"amzn_l1.png", dpi=110)
feat[["mid","spread_ticks","bid_qty","ask_qty"]].describe().round(2)

In [ ]:
# Throughput of the pure C++ replay from Python (includes numpy hand-off)
for t in ["AMZN","MSFT"]:
    m = D.load_messages(t)
    t0 = time.perf_counter(); r = lob.replay(*m.as_args()); dt = time.perf_counter()-t0
    print(f"{t}: {len(m):,} msgs in {dt*1e3:.1f} ms -> {len(m)/dt/1e6:.1f} M msgs/s")